In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

For questions 1 and 2: 

Given a dataset with time series data containing an event, use a linear regression to test whether there was a discontinuity in the data at the event. Consider the possibility, first, of a discontinuity only in the value of the variable but not the derivative. Then consider that there may be a discontinuity in the first derivative (the slope).  
Use the file homework_3.1.csv. 

## Question 1   

Which dataset is most likely to have a discontinuity (or has the strongest discontinuity) in the value at the event (at time = 50)? 

- A: value2
 
- B: value3 

- C: value1 

In [2]:
df1 = pd.read_csv('data/homework_3.1.csv')
df1.head()

,time,value1,value2,value3
0,0,1.764052,1.883151,-0.369182
1,1,0.420157,-1.327759,-0.219379
2,2,1.018738,-1.230485,1.139660
3,3,2.300893,1.029397,0.715264
4,4,1.947558,-1.093123,0.720132


In [3]:
df1["after_50"] = (df1["time"] >= 50).astype(int)
df1.head()

,time,value1,value2,value3,after_50
0,0,1.764052,1.883151,-0.369182,0
1,1,0.420157,-1.327759,-0.219379,0
2,2,1.018738,-1.230485,1.139660,0
3,3,2.300893,1.029397,0.715264,0
4,4,1.947558,-1.093123,0.720132,0


In [4]:
results = {}
for cols in ["value1", "value2", "value3"]:
    X = df1[["time", "after_50"]]
    y = df1[cols]
    model = sm.OLS(y, sm.add_constant(X)).fit()
    results[cols] = { "after_50_coef": model.params["after_50"],
                      "p_value": model.pvalues["after_50"], 
                      "t_value": model.tvalues["after_50"] }
results_df = pd.DataFrame(results).T
results_df

,after_50_coef,p_value,t_value
value1,0.850813,0.085607,1.736745
value2,0.682746,0.113103,1.598864
value3,1.767254,0.000033,4.356868


Answer : B
___

## Question 2

Which dataset is most likely to have a discontinuity (or has the strongest discontinuity) in the derivative (at time = 50)? 

- A: value2

- B: value3

- C: value1


In [5]:
df1["event_time"] = (df1["time"] - 50) * df1["after_50"]
results_event = {}
for col in ["value1", "value2", "value3"]:
    X = df1[["time", "after_50", "event_time"]]
    y = df1[col]
    model = sm.OLS(y, sm.add_constant(X)).fit()
    results_event[col] = { "after_50_coef": model.params["after_50"],
                            "p_value_after_50": model.pvalues["after_50"],
                            "t_value_after_50": model.tvalues["after_50"],
                            "event_time_coef": model.params["event_time"],
                            "p_value_event_time": model.pvalues["event_time"],
                            "t_value_event_time": model.tvalues["event_time"]}
results_event_df = pd.DataFrame(results_event).T
results_event_df

,after_50_coef,p_value_after_50,t_value_after_50,event_time_coef,p_value_event_time,t_value_event_time
value1,0.903475,0.020170,2.362488,0.105325,3.602073e-12,7.951259
value2,0.701206,0.094565,1.688480,0.036921,1.181130e-02,2.566732
value3,1.792602,0.000008,4.724217,0.050695,2.076610e-04,3.857133


Answer : C
___

For questions 3 to 5:  

Given a dataset with treatment and control data having “before” and “after” parts, apply a differences-in-differences regression.  
Use homework_3.2.a.csv and homework_3.2.b.csv. 

## Question 3


Which dataset likely has the largest treatment effect, assuming that the treatment and control groups have parallel trends? 

- A: Group 1

- B: Group 2

In [6]:
g1 = pd.read_csv('data/homework_3.2.a.csv')
g2 = pd.read_csv('data/homework_3.2.b.csv')
g1.head()

,group1,time1,outcome1
0,0,0,0.882026
1,0,1,1.600079
2,0,0,0.489369
3,0,1,2.520447
4,0,0,0.933779


In [7]:
g2.head()

,group2,time2,outcome2
0,0,0,0.667155
1,0,1,2.470969
2,0,0,-0.506778
3,0,1,1.525657
4,0,0,0.273664


In [8]:
g1["group1_time1"] = g1["group1"] * g1["time1"]
g2["group2_time2"] = g2["group2"] * g2["time2"]

X1 = g1[["group1", "time1", "group1_time1"]]
Y1 = g1["outcome1"]

X2 = g2[["group2", "time2", "group2_time2"]]
Y2 = g2["outcome2"]

model1 = sm.OLS(Y1, sm.add_constant(X1)).fit()
model2 = sm.OLS(Y2, sm.add_constant(X2)).fit()


summary_result = pd.DataFrame({
    "const": [model1.params["const"], model2.params["const"]],
    "group": [model1.params["group1"], model2.params["group2"]],
    "time": [model1.params["time1"], model2.params["time2"]],
    "group_time": [model1.params["group1_time1"],model2.params["group2_time2"]]
}, index=["Group 1", "Group 2"])

summary_result

,const,group,time,group_time
Group 1,-0.025849,1.986278,1.427213,0.685847
Group 2,0.102107,1.847719,1.265513,1.349859


Answer : B
___

## Question 4

Using the standard errors for regression, which dataset has the most statistically significant (and nonzero) treatment effect? 

- A: Group 2

- B: Group 1

In [11]:
significance_result = pd.DataFrame({
    "treatment_effect": [model1.params["group1_time1"],model2.params["group2_time2"]],
    "std_error": [model1.bse["group1_time1"],model2.bse["group2_time2"]],
    "t_value": [model1.tvalues["group1_time1"],model2.tvalues["group2_time2"]],
    "p_value": [model1.pvalues["group1_time1"],model2.pvalues["group2_time2"]]
}, index=["Group 1", "Group 2"])

significance_result

,treatment_effect,std_error,t_value,p_value
Group 1,0.685847,0.062522,10.969611,1.640050e-26
Group 2,1.349859,0.147047,9.179774,2.432436e-19


In [9]:
model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               outcome1   R-squared:                       0.899
Model:                            OLS   Adj. R-squared:                  0.899
Method:                 Least Squares   F-statistic:                     2964.
Date:                Thu, 17 Sep 2026   Prob (F-statistic):               0.00
Time:                        18:19:51   Log-Likelihood:                -712.28
No. Observations:                1000   AIC:                             1433.
Df Residuals:                     996   BIC:                             1452.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0258      0.031     -0.829      0.408      -0.087       0.035
group1           1.9863      0.044     44.928      0.000       1.900       2.073
time1            1.4272      0.044     32.315      0.000       1.341       1.514
group1_time1     0.6858      0.063     10.970      0.000       0.563       0.809
==============================================================================
Omnibus:                        0.196   Durbin-Watson:                   2.048
Prob(Omnibus):                  0.907   Jarque-Bera (JB):                0.236
Skew:                           0.032   Prob(JB):                        0.889
Kurtosis:                       2.961   Cond. No.                         6.85
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [10]:
model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               outcome2   R-squared:                       0.663
Model:                            OLS   Adj. R-squared:                  0.662
Method:                 Least Squares   F-statistic:                     653.6
Date:                Thu, 17 Sep 2026   Prob (F-statistic):          9.62e-235
Time:                        18:19:58   Log-Likelihood:                -1567.5
No. Observations:                1000   AIC:                             3143.
Df Residuals:                     996   BIC:                             3163.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.1021      0.073      1.392      0.164      -0.042       0.246
group2           1.8477      0.104     17.770      0.000       1.644       2.052
time2            1.2655      0.104     12.183      0.000       1.062       1.469
group2_time2     1.3499      0.147      9.180      0.000       1.061       1.638
==============================================================================
Omnibus:                        0.711   Durbin-Watson:                   2.069
Prob(Omnibus):                  0.701   Jarque-Bera (JB):                0.796
Skew:                           0.047   Prob(JB):                        0.672
Kurtosis:                       2.899   Cond. No.                         6.85
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Answer : B
___

## Question 5

Which of these is closest to the treatment effect for group 2? 

- A: 4.209

- B: 0.5831

- C: 1.248 



Answer : C
___